<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Yield Prediction in Semiconductor Wafer Fab
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style = 'font-size:20px;font-family:Arial'><b>Introduction</b></p>
<p style = 'font-size:16px;font-family:Arial'>In modern semiconductor manufacturing, production processes are monitored through a large network of sensors and measurement systems that continuously capture signals from various stages of the fabrication pipeline. These signals provide valuable insights into equipment performance, process conditions, and product quality. However, the large volume of data generated often includes a mixture of meaningful signals, redundant information, and noise. As a result, process engineers are frequently faced with far more variables than are necessary for effective monitoring and analysis, making it challenging to identify which signals truly influence production outcomes.<br>
To address this challenge, feature selection techniques can be applied to systematically identify the most relevant signals from the large pool of collected data. By treating each signal as a feature, advanced analytics can be used to evaluate and determine which variables have the strongest relationship with yield outcomes. The selected signals enable process engineers to better understand the factors contributing to yield excursions and production variability. Leveraging these insights allows organizations to predict yield types more accurately, improve process throughput, shorten learning cycles, and reduce overall manufacturing costs, ultimately leading to more efficient and reliable semiconductor production.</p>
<p style = 'font-size:18px;font-family:Arial'><b>Why Vantage?</b></p>  
<p style = 'font-size:16px;font-family:Arial'>
Teradata Vantage enables semiconductor manufacturers to efficiently analyze large volumes of sensor and process data by performing advanced analytics and machine learning directly within the database. By leveraging in-database processing, organizations can apply feature selection techniques to identify the most critical signals influencing yield outcomes while minimizing data movement and accelerating analysis. This allows process engineers to quickly uncover key factors behind yield variations, improve predictive capabilities, and optimize manufacturing processes to enhance throughput and reduce production costs.</p>
<p style = 'font-size:18px;font-family:Arial'><b>About Dataset</b></p>
<p style = 'font-size:16px;font-family:Arial'>The dataset presented in this case represents a selection of such features where each example represents a single production entity with associated measured features and the labels represent a simple pass/fail yield for in house line testing and associated date time stamp for that specific test point.<br>
Using feature selection techniques it is desired to rank features according to their impact on the overall yield for the product, causal relationships may also be considered with a view to identifying the key features.
    <br>SECOM Dataset: <a href = 'https://www.kaggle.com/datasets/paresh2047/uci-semcom/data'>Data Source</a> <ul style = 'font-size:16px;font-family:Arial'><li>1567 examples 591 features</li><li> 104 fails in dataset</li><li> We have added 1 column for id and 1 column for fail/pass label</li></ul>
    <br>

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>1.Connect to Vantage, Import python packages and explore the dataset</b></p>


In [ ]:
import getpass
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

# import all Teradataml functions and supporting libraries
from teradataml import *

configure.val_install_location = "VAL"
display.max_rows = 5

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial'><b>1.1 Connect to Vantage</b></p>
<p style = 'font-size:16px;font-family:Arial'>You will be prompted to provide the password. Enter your password, press the Enter key, and then use the down arrow to go to the next cell.</p>

In [ ]:
%run -i ../startup.ipynb
eng = create_context(host = 'host.docker.internal', username='demo_user', password = password)
print(eng)

In [ ]:
%%capture
execute_sql('''SET query_band='DEMO=PP_Yield_Prediction_Semiconductor_Python.ipynb;' UPDATE FOR SESSION; ''')

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>1.2 Getting Data for This Demo</b></p>

<p style = 'font-size:16px;font-family:Arial'>We have provided data for this demo on cloud storage. We have the option of either running the demo using foreign tables to access the data without using any storage on our environment or downloading the data to local storage, which may yield some what faster execution. However, we need to consider available storage. There are two statements in the following cell, and one is commented out. We may switch which mode we choose by changing the comment string.</p></p>

In [ ]:
#%run -i ../run_procedure.py "call get_data('DEMO_Secom_cloud');"
 # takes about 30 seconds, estimated space: 0 MB
%run -i ../run_procedure.py "call get_data('DEMO_Secom_local');" 
# takes about 5 minutes, estimated space: 6 MB

In [ ]:
%run -i ../run_procedure.py "call space_report();"

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>2. Data Preparation</b></p>
<p style = 'font-size:16px;font-family:Arial'> For using the data for yield prediction, we will do below preprocessing steps.
 <ol style = 'font-size:16px;font-family:Arial'> 
     <li>Read data table as teradata dataframe</li>
     <li>Inspect the column metadata using ColumnSummary</li>
     <li>Remove columns that have constant value</li>
     <li>Remove columns that have low correlation (absolute correlation less than 5% with target label)</li>
     <li>Split data into train and test set with TrainTestSplit</li>
     <li>Normalize train and test with Standard scaler</li>
     <li>Upsampling minor class with SMOTE to balance training data set</li>
</ol>     
       

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>2.1 Read data in Teradataml dataframe</b></p>


In [ ]:
tdf = DataFrame(in_schema('DEMO_Secom', 'Secom_Data'))
tdf

In [ ]:
tdf.shape

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>2.2 View Column information</b></p>

<p style = 'font-size:16px;font-family:Arial'>ColumnSummary provides more details on column values and ranges. Note that the resulting DataFrame is a property of the function object.</p>

In [ ]:
obj = ColumnSummary(data=tdf, target_columns=[':'])
obj.result

<p style = 'font-size:16px;font-family:Arial'>Check number of columns that have more than 50% null values.

In [ ]:
colsummary = obj.result
colsummary50 = colsummary[colsummary['NullPercentage'] >= 50].select('ColumnName')
colsummary50

In [ ]:
colsummary50.shape #number of columns with null% >= 50%

<p style = 'font-size:16px;font-family:Arial'>As we can see from above we have 28 columns which contain more than 50% null values. We will drop these columns from our prediction dataset.

In [ ]:
collist=colsummary50.get_values().flatten().tolist()
tdf50 = tdf.drop(columns= collist)

In [ ]:
tdf50.info(verbose=False) # 593-28 = 565

<p style = 'font-size:16px;font-family:Arial'> For rest of the columns which have null values we will fill columns's null values using 0.</p>

In [ ]:
tdf50 = tdf50.fillna(0)

In [ ]:
obj = GetRowsWithMissingValues(data=tdf50)
obj.result # confirmed no missing values after fillna


<p style = 'font-size:16px;font-family:Arial'> From above we can confirm that there is no null value in the dataset now.</p> 

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>2.3 Drop columns that have constant values</b></p>



In [ ]:
tdf50std = tdf50.describe(statistics = 'std')
tdf50std

In [ ]:
tdf50std0 = tdf50std[tdf50std['StatValue'] == 0.0]
tdf50std0 # 112 columns have constant values

<p style = 'font-size:16px;font-family:Arial'>Drop columns with constant single value.

In [ ]:
zero_std_columns = tdf50std0.select('Attribute').get_values().flatten().tolist()
tdf50std0drop = tdf50.drop(columns= zero_std_columns)
tdf50std0drop.info(verbose=False) # 565-112=453

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>2.4 Drop columns that have low correlation with target label</b></p>
<p style = 'font-size:16px;font-family:Arial'>Calculate corralation matrix using teradataml Vantage Analytics Library Functions(VAL).

In [ ]:
cols = tdf50std0drop.columns
cols.remove("coltime")

In [ ]:
obj = valib.Matrix(data=tdf50std0drop,
                       columns=cols,                     
                       type="COR")

In [ ]:
tdf50std0drop_cor=obj.result
tdf50std0drop_cor

In [ ]:
tdf50std0drop_cor.shape

<p style = 'font-size:16px;font-family:Arial'>Check how many columns with absolute correlation to Label less than 5%. 

In [ ]:
tdf50std0drop_cor005 = tdf50std0drop_cor[tdf50std0drop_cor.collabel.abs()<0.05]
tdf50std0drop_cor005.shape #We have 371 features with low correlation 0.05 with label

<p style = 'font-size:16px;font-family:Arial'>Drop columns with low correlation to Label.

In [ ]:
cols= tdf50std0drop_cor005.select('rowname').get_values().flatten().tolist()
tdf50std0cor005drop = tdf50std0drop.drop(columns= cols)
tdf50std0cor005drop.shape # # 565-112=453 -371 = 82

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>2.5 Split into Train/Test sets</b></p>


In [ ]:
target = "collabel"
key = "myrow_id"

In [ ]:
# split in training & test set 80 20 stratified
TrainTestSplit_out = TrainTestSplit(
                                    data = tdf50std0cor005drop,
                                    id_column = key,
                                    stratify_column=target, 
                                    seed = 42, 
                                    train_size = 0.8
                                   )


In [ ]:
# Split into 2 virtual dataframes
df_train = TrainTestSplit_out.result[TrainTestSplit_out.result['TD_IsTrainRow'] == 1].drop(['TD_IsTrainRow'], axis = 1)
df_test = TrainTestSplit_out.result[TrainTestSplit_out.result['TD_IsTrainRow'] == 0].drop(['TD_IsTrainRow'], axis = 1)


In [ ]:
print(df_train.shape,df_test.shape)

In [ ]:
copy_to_sql(
    df          = df_train,
    schema_name = "demo_user",
    table_name  = "secomtrain",
    primary_index = "myrow_id",
    if_exists   ='replace'
)

In [ ]:
copy_to_sql(
    df          = df_test,
    schema_name = "demo_user",
    table_name  = "secomtest",
    primary_index = "myrow_id",
    if_exists   ='replace'
)

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>2.5 Scale Train and Test sets</b></p>
<p style = 'font-size:16px;font-family:Arial'><b>ScaleFit and ScaleTransform </b>scales specified input
table columns i.e perform the specific scale methods like standard deviation, mean etc to the input columns </p>
<p style = 'font-size:16px;font-family:Arial'><b>Train Set<b>

In [ ]:
secom_train = DataFrame(in_schema('demo_user', 'secomtrain'))

In [ ]:
colscale = secom_train.columns
## exclude time label id
colscale.remove("coltime")
colscale.remove("collabel")
colscale.remove("myrow_id")

In [ ]:
fit_obj = ScaleFit(data= secom_train,
                    target_columns= colscale, 
                    scale_method="STD")

In [ ]:
# Scale values specified in the input data using the fit data generated by the Scale() function above.
obj = ScaleTransform(data= secom_train,
                     object=fit_obj.output,
                     accumulate= ['myrow_id', 'coltime', 'collabel'])

In [ ]:
secomtrain_scaled=obj.result
secomtrain_scaled

<p style = 'font-size:16px;font-family:Arial'><b>Test Set<b>

In [ ]:
secomtest = DataFrame(in_schema('demo_user', 'secomtest'))

In [ ]:
# Scale values specified in the input data using the fit data generated by the Scale() function above.
obj = ScaleTransform(data= secomtest,
                     object=fit_obj.output,
                     accumulate= ['myrow_id', 'coltime', 'collabel'])


In [ ]:
secomtest_scaled=obj.result
secomtest_scaled

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>2.6 Upsampling to balance data  - SMOTE on scaled train set</b></p>
<p style = 'font-size:16px;font-family:Arial'>The data has very unbalanced Pass/Fail (0/1) ratio, we will use SMOTE function to upsampling the minor class to balance the data for model training.</p>


In [ ]:
print(secomtrain_scaled.select(['myrow_id','collabel']).groupby('collabel').count())


In [ ]:
colsmote = secomtrain_scaled.columns
## exclude time label id
colsmote.remove("coltime")
colsmote.remove("collabel")
colsmote.remove("myrow_id")

In [ ]:
# teradata function to Generate synthetic samples using smote algorithm.
smote_out = SMOTE(data = secomtrain_scaled,
                  id_column= key, #'myrow_id',
                  minority_class='1',
                  response_column= target, #'collabel',
                  input_columns = colsmote, 
                  oversampling_factor= 14, 
                  sampling_strategy='smote',
                  seed=10)



In [ ]:
smote_out.result.shape

In [ ]:
secomtrain_scaled_smote = smote_out.result
secomtrain_scaled_smote

<p style = 'font-size:16px;font-family:Arial'>Append the major class into a Train full set.

In [ ]:
secomtrain_scaled_smote_full = secomtrain_scaled.loc[secomtrain_scaled[target] == 0].concat(secomtrain_scaled_smote)

In [ ]:
secomtrain_scaled.loc[secomtrain_scaled[target] == 0].shape 

In [ ]:
secomtrain_scaled_smote_full.shape # 1162 + 1170 = 2332

<p style = 'font-size:16px;font-family:Arial'>Generate smoterow_id for Train full set.

In [ ]:
from sqlalchemy import func

# Create row number column in the DataFrame.
secomtrain_scaled_smote_fullidx = secomtrain_scaled_smote_full.assign(
    smoterow_id = func.row_number().over(order_by= secomtrain_scaled_smote_full.myrow_id.expression.asc()))


In [ ]:
secomtrain_scaled_smote_fullidx.shape

In [ ]:
copy_to_sql(
    df          = secomtrain_scaled_smote_fullidx,
    schema_name = "demo_user",
    table_name  = "secomtrain_scaled_smote_fullidx",
    primary_index = "smoterow_id",
    if_exists   ='replace'
)

In [ ]:
secomtrain_scaled_smote_fullidx = DataFrame(in_schema('demo_user', 'secomtrain_scaled_smote_fullidx'))

In [ ]:
smoterow_id_tmin=secomtrain_scaled_smote_fullidx.select('smoterow_id').min()
smoterow_id_tmax=secomtrain_scaled_smote_fullidx.select('smoterow_id').max() 
print(smoterow_id_tmin,smoterow_id_tmax)

<p style = 'font-size:16px;font-family:Arial'>Now we can see Pass/Fail (0/1) ratio is close to 1.

In [ ]:
print(secomtrain_scaled_smote_fullidx.select(['smoterow_id','collabel']).groupby('collabel').count())


<p style = 'font-size:16px;font-family:Arial' >Generate smoterow_id for Test set for consistancy. 

In [ ]:
secomtest_scaled

In [ ]:
from sqlalchemy import func

# Create row number column in the DataFrame.
secomtest_scaled_idx = secomtest_scaled.assign(
    smoterow_id = func.row_number().over(order_by= secomtest_scaled.myrow_id.expression.asc()))

In [ ]:
incr = smoterow_id_tmax.get_values()
# setting test id start after train id
secomtest_scaled_idx = secomtest_scaled_idx.assign(
    smoterow_id = secomtest_scaled_idx.smoterow_id + incr)

In [ ]:
copy_to_sql(
    df          = secomtest_scaled_idx,
    schema_name = "demo_user",
    table_name  = "secomtest_scaled_idx",
    primary_index = "smoterow_id",
    if_exists   ='replace'
)

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>3. Model Training</b></p>
<p style = 'font-size:18px;font-family:Arial'><b> DecisionForest Model </b>

In [ ]:
secomtrain_scaled_smote_fullidx = DataFrame(in_schema('demo_user', 'secomtrain_scaled_smote_fullidx'))

In [ ]:
colsmoteidx = secomtrain_scaled_smote_fullidx.columns
## exclude time label id smoterow_id
colsmoteidx.remove("coltime")
colsmoteidx.remove("collabel")
colsmoteidx.remove("myrow_id")
colsmoteidx.remove("smoterow_id")

In [ ]:
colsmoteidx

In [ ]:
%%time
rft_model = DecisionForest(data= secomtrain_scaled_smote_fullidx,
                           input_columns= colsmoteidx,
                           response_column= "collabel", 
                           tree_type= "classification",
                           min_impurity= 0.0,
                           min_node_size= 1,
                           max_depth=20,
                           seed=42)


<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>4. Model Explain</b></p>
<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial'><b> 4.1 util functions </b>

In [ ]:
def compute_feature_importance(feat_df):
    df = feat_df.to_pandas()
    df = df.T.reset_index()
    df=df.rename(columns={'index': 'Feature', 0: 'Importance'})
    df['Feature'] = df['Feature'].str.replace('TD_', '')
    df['Feature'] = df['Feature'].str.replace('_SHAP', '')
    return df  

def plot_feature_importance(df, img_filename):
    df = df.sort_values(by="Importance", ascending=False)
    # Plot the bar graph
    plt.figure(figsize=(5, 10))
    sns.barplot(x="Importance",y="Feature",data=df, palette="viridis")
    plt.title("Feature Importance")
    plt.xlabel("SHAP Importance Value")
    plt.ylabel("Features")
    plt.tight_layout()
    plt.show()
    plt.clf()
    

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>4.2  Find Feature Importance using SHAP</b></p>

In [ ]:
%%time
rft_model_shap = Shap(data= secomtrain_scaled_smote_fullidx, 
                        input_columns= colsmoteidx, 
                        id_column='smoterow_id',
                        object=rft_model.result, 
                        model_type="Classification",
                        training_function="TD_DECISIONFOREST",              
                        detailed=True)

feat_df = rft_model_shap.output_data

# about 5mins to run 

In [ ]:
df = compute_feature_importance(feat_df)
plot_feature_importance(df, "feature_importance")

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>5. Model Predict</b></p>
<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial'><b> 5.1 Prediction on Train set </b>

In [ ]:
secomtrain_scaled_smote_fullidx = DataFrame(in_schema('demo_user', 'secomtrain_scaled_smote_fullidx'))

In [ ]:
rft_pred_train = TDDecisionForestPredict(object = rft_model,
                                        newdata = secomtrain_scaled_smote_fullidx,
                                        id_column = "smoterow_id",
                                        object_order_column=['task_index', 'tree_num'],
                                        output_prob = True,
                                        output_responses = ['0','1'],
                                        accumulate="collabel")



In [ ]:
rft_pred_train

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial'><b> 5.2 Prediction on Test set </b>

In [ ]:
secomtest_scaled_idx = DataFrame(in_schema('demo_user', 'secomtest_scaled_idx'))

In [ ]:
rft_pred_test = TDDecisionForestPredict(object = rft_model,
                                        newdata = secomtest_scaled_idx,
                                        id_column = "smoterow_id",
                                        object_order_column=['task_index', 'tree_num'],
                                        output_prob = True,
                                        output_responses = ['0','1'],
                                        accumulate="collabel")



In [ ]:
rft_pred_test

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>6. Model Evaluate</b></p>
<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial'><b> 6.1 util functions </b>

In [ ]:

# Define function to plot a confusion matrix from given data
def plot_confusion_matrix(cf, img_filename):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.matshow(cf, cmap=plt.cm.Blues, alpha=0.3)
    for i in range(cf.shape[0]):
        for j in range(cf.shape[1]):
            ax.text(x=j, y=i,s=cf[i, j], va='center', ha='center', size='xx-large')
    ax.set_xlabel('Predicted labels');
    ax.set_ylabel('True labels'); 
    ax.set_title('Confusion Matrix');
    plt.show() 
    plt.clf()



<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial'><b> 6.2 ClassificationEvaluator </b>

In [ ]:
# on test set
ClassificationEvaluator_obj = ClassificationEvaluator(
    data= rft_pred_test.result,
    observation_column= 'collabel',
    prediction_column='Prediction',
    num_labels=2
)

<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial'><b> 6.3 Confusion Matrix </b>

In [ ]:
from sklearn.metrics import confusion_matrix

cm_df = ClassificationEvaluator_obj.result
cm_df = cm_df.select(['CLASS_1','CLASS_2'])
cm_df_t = cm_df.to_pandas().T

cm = confusion_matrix(rft_pred_test.result.to_pandas()['collabel'], rft_pred_test.result.to_pandas()['prediction'])

plot_confusion_matrix(cm_df_t.values, "confusion_matrix")


<hr style='height:1px;border:none;'>
<p style = 'font-size:18px;font-family:Arial'><b> 6.4 ROC </b>

In [ ]:
# Generate and save ROC curve plot
roc_out = ROC(
    data=rft_pred_test.result,
    probability_column='prob_1',
    observation_column= 'collabel',
    positive_class='1',
    num_thresholds=50 
)
roc_out_pd = roc_out.output_data.to_pandas()
auc = roc_out.result.get_values()[0][0]

In [ ]:

plt.plot(roc_out_pd['fpr'],roc_out_pd['tpr'], label=f'ROC (AUC = {round(auc, 2)})')
plt.plot(roc_out_pd['tpr'],roc_out_pd['tpr'], label='Baseline (AUC = 0.5)')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.title('ROC Curve')
plt.grid(True)
plt.legend(loc='lower right')
plt.show()


<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>7. Cleanup</b></p>
<p style = 'font-size:18px;font-family:Arial'><b>Work Tables</b></p>
<p style = 'font-size:16px;font-family:Arial;'>
We need to clean up our work tables to prevent errors next time.

In [ ]:
tables = ['secomtrain','secomtest','secomtrain_scaled_smote_fullidx','secomtest_scaled_idx']

# Loop through the list of tables and execute the drop table command for each table
for table in tables:
    try:
        db_drop_table(table_name = table)
    except:
        pass

<p style = 'font-size:18px;font-family:Arial'><b>Databases and Tables</b></p>
<p style = 'font-size:16px;font-family:Arial'>We will use the following code to clean up tables and databases created for this demonstration.</p>

In [ ]:
%run -i ../run_procedure.py "call remove_data('DEMO_Secom');" 
#Takes 10 seconds

In [ ]:
remove_context()

<footer style="padding-bottom:35px; border-bottom:3px solid #91A0Ab">
    <div style="float:left;margin-top:14px">ClearScape Analytics™</div>
    <div style="float:right;">
        <div style="float:left; margin-top:14px">
            Copyright © Teradata Corporation - 2026. All Rights Reserved
        </div>
    </div>
</footer>